# Vibronic coupling in tetracene dimers

## General post-processing / prototyping notebook

### To be done on stanage:
- geometry optimisation (optional)
- MO coefficients (save checkfile)
- EPH coupling (save hdf5 with ephmat, omega, modevec)

### Can be done locally:
- Boys localisation and partial diagonalisation (use populations from checkfile)
- orbital visualisation
- transform EPH coupling to localised subspace

### Functions / modules we need:
- IN: checkfile -  OUT: mol, scf object
- mo_frontier_Boys, fontier_orbital_idx = frontier_Boys(mf)
- 
- etc...

In [1]:
from pyscf import gto, dft, lo, eph, lib
from pyscf.tools import cubegen
import numpy as np
import time, h5py, argparse
import py3Dmol
import matplotlib.pyplot as plt

In [127]:
dir = 'runs/10208100' # dimer_opt
checkfile = dir + '/calc.chk' 

#checkfile = 'calc.chk' # water

mol = lib.chkfile.load_mol(checkfile)
mf = dft.RKS(mol)
mf.__dict__.update(lib.chkfile.load(checkfile, "scf"))

In [128]:
# populations

# --- Identify frontier orbital indices ---
mo_occ = mf.mo_occ
nocc = np.count_nonzero(mo_occ > 0)
homo = nocc - 1
lumo = nocc
frontier_idx = [homo-1, homo, lumo, lumo+1]

# --- Localize frontier orbitals ---
mo_coeff = mf.mo_coeff
mo_front = mo_coeff[:, frontier_idx]
mo_front_Boys = lo.Boys(mol, mo_front).kernel()
mo_Boys = mo_coeff.copy()
mo_Boys[:,frontier_idx] = mo_front_Boys

# --- Atomic populations of localized orbitals ---
pop_front_Boys = np.zeros((mol.natm, len(frontier_idx)))
labels = mol.ao_labels()
for i in range(len(frontier_idx)):
    C = mo_front_Boys[:,i]
    dm_mo = np.outer(C, C)
    pop, _ = mf.mulliken_pop(mol, dm_mo, verbose=0)
    for ia in range(mol.natm):
        idx = [k for k, lbl in enumerate(labels) if int(lbl.split()[0]) == ia]
        atom_prob = np.sum(pop[idx])
        pop_front_Boys[ia, i] = atom_prob

In [129]:
# --- create xyz string from atomic symbols and coordinates ---
def xyz_string(symbls, coords):
    xyz = f"{len(symbls)}\n\n"
    for s, (x,y,z) in zip(symbls, coords):
        xyz += f"{s} {x} {y} {z}\n"
    return xyz

### Determine which localized orbital reside on which fragment of the molecule, and reorder them so they appear in the following order:
1. <span style="color:blue">**idx1**</span>    left front 
2. <span style="color:red">**idx2**</span>     left back
3. <span style="color:green">**idx3**</span>   right front
4. <span style="color:yellow">**idx4**</span>  right back

This relies on Boys-localised orbitals being localised on these fragments. Always visualise orbitals to make sure that's correct.

In [130]:
# --- Find atom indices corresponding to localized orbitals ---

coords = mol.atom_coords() * 0.529177
symbols = mol.elements
x, y, z = coords.T
x0 = np.mean(x)
y0 = np.mean(y)
dx = .5
dy = .1

idx1 = (x < x0 - dx) & (y < y0 - dx)
coords1 = coords[idx1]
symbols1 = np.array(symbols)[idx1]

idx2 = (x < x0 - dx) & (y > y0 + dx)
coords2 = coords[idx2]
symbols2 = np.array(symbols)[idx2]

idx3 = (x > x0 + dx) & (y < y0 - dx)
coords3 = coords[idx3]
symbols3 = np.array(symbols)[idx3]

idx4 = (x > x0 + dx) & (y > y0 + dx)
coords4 = coords[idx4]
symbols4 = np.array(symbols)[idx4]


view = py3Dmol.view(width=600, height=300)
view.clear()
view.addModel(xyz_string(symbols1, coords1), 'xyz', {})
view.addModel(xyz_string(symbols2, coords2), 'xyz', {})
view.addModel(xyz_string(symbols3, coords3), 'xyz', {})
view.addModel(xyz_string(symbols4, coords4), 'xyz', {})

view.setStyle({'model':0, 'not':{'elem':'H'}}, {'stick':{'colorscheme':'blueCarbon'}})
view.setStyle({'model':1, 'not':{'elem':'H'}}, {'stick':{'colorscheme':'redCarbon'}})
view.setStyle({'model':2, 'not':{'elem':'H'}}, {'stick':{'colorscheme':'greenCarbon'}})
view.setStyle({'model':3, 'not':{'elem':'H'}}, {'stick':{'colorscheme':'yellowCarbon'}})

view.addModel(xyz_string(symbols, coords), 'xyz', {})
view.setStyle({'model':4,}, {'stick':{'colorscheme':'grayCarbon'}})

view.setViewStyle({"style": "outline"})

view.setBackgroundColor('white')
view.zoomTo()
view.rotate(-45, 'x')
view.zoom(1.5)
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [131]:
# --- Reorder localised orbitals ---
fragment_atom_idx = np.array([idx1, idx2, idx3, idx4], dtype=int)
fragment_Boys_overlap = fragment_atom_idx @ pop_front_Boys
idx1234 = np.argmax(fragment_Boys_overlap, axis=1)
mo_front_Boys = mo_front_Boys.copy()[:,idx1234]
mo_Boys = mo_coeff.copy()
mo_Boys[:,frontier_idx] = mo_front_Boys

In [8]:
# --- Export frontier orbitals as cube files ---
for i, orb_idx in enumerate(frontier_idx):
    cubegen.orbital(mol, 
                    'orbBoys' + str(i+1) + '.cube', 
                    mo_Boys[:, orb_idx],
                    #mo_front_Boys[:, i],  
                    nx=400, ny=200, nz=200)

### Visualise localised orbitals

In [9]:
# --- Import and visualise ---
# Set html=True for visualisation in a web browser.
# This keeps the notebook lightweight and running.

def view_orbital(cube_filename, mol_obj, iso=0.025, alpha=0.9, html=True):
    with open(cube_filename) as f:
        cube_data = f.read()
    view = py3Dmol.view(width=600, height=300)
    symbols = mol_obj.elements
    coords = mol_obj.atom_coords() * 0.529177
    view.addModel(xyz_string(symbols, coords), 'xyz')
    view.setStyle({'model':0, 'not':{'elem':'H'}}, {'stick':{'colorscheme':'grayCarbon'}})
    view.setStyle({'model':0}, {'stick':{'colorscheme':'grayCarbon'}})
    view.addVolumetricData(cube_data, "cube", {
        "isoval": iso,
        "color": "blue",
        "opacity": alpha
    })
    view.addVolumetricData(cube_data, "cube", {
        "isoval": -iso,
        "color": "red",
        "opacity": alpha
    })
    view.setViewStyle({"style": "outline"})
    view.setBackgroundColor('white')
    #view.setBackgroundColor('rgba(0,0,0,0)')
    view.zoomTo()
    view.rotate(-45, 'x')
    view.zoom(1.5)

    if html:
        html = view._make_html()
        html_filename = cube_filename.split('.')[0] + '.html'
        with open(html_filename, "w") as f:
            f.write(html)
        import webbrowser, os
        webbrowser.open('file://' + os.path.realpath(html_filename))

    return view

In [10]:
for orbName in ['orbBoys1.cube', 'orbBoys2.cube', 'orbBoys3.cube', 'orbBoys4.cube']:
    v = view_orbital(orbName, mol, iso=0.02, alpha=0.9, html=True)

### Fock matrix in the frontier orbital subspace

In [132]:
# --- Fock matrix in the localised basis ---
dm = mf.make_rdm1()
Fao = mf.get_fock(dm=dm)
FBoys = mo_front_Boys.T @ Fao @ mo_front_Boys
print(FBoys)

[[-0.13888843  0.02607784 -0.00230496 -0.00103608]
 [ 0.02607784 -0.13888842 -0.00103614 -0.00230464]
 [-0.00230496 -0.00103614 -0.13888758  0.02607799]
 [-0.00103608 -0.00230464  0.02607799 -0.13888763]]


In [134]:
# --- Block diagonalisation ---
FLeft = FBoys[:2,:2]
FRight = FBoys[-2:,-2:]
def eigsort(M):
    val, vec = np.linalg.eig(M)
    i = np.argsort(val)
    return val[i], vec[:,i]
valLeft,  vecLeft = eigsort(FLeft)
valRight, vecRight = eigsort(FRight)
UBoys2loc = np.block([[vecLeft,np.zeros((2,2))],[np.zeros((2,2)), vecRight]])
mo_front_block = mo_front_Boys @ UBoys2loc
print(UBoys2loc.T @ FBoys @ UBoys2loc)

[[-1.64966269e-01 -4.16333634e-17 -1.26868915e-03  1.26575380e-07]
 [-4.85722573e-17 -1.12810582e-01 -1.90692996e-07  3.34090919e-03]
 [-1.26868915e-03 -1.90692996e-07 -1.64965600e-01 -5.55111512e-17]
 [ 1.26575380e-07  3.34090919e-03 -8.32667268e-17 -1.12809616e-01]]


In [115]:
#print('Fock matrix in the MO basis (invcm)')
#print(mo_front.T @ Fao @ mo_front * 219474.63)
#print('')
#print('Fock matrix in the Boys basis (invcm)')
#print(FBoys * 219474.63)
#print('')
#print('Fock matrix in the block-diagonal basis (invcm)')
#print(UBoys2loc.T @ FBoys @ UBoys2loc * 219474.63)

In [14]:
# --- Export block-localised orbitals as cube files ---
for i, orb_idx in enumerate(frontier_idx):
    cubegen.orbital(mol, 
                    'orbBlock' + str(i+1) + '.cube', 
                    mo_front_block[:, i],
                    nx=400, ny=200, nz=200)

In [141]:
for orbName in ['orbBlock1.cube', 'orbBlock2.cube', 'orbBlock3.cube', 'orbBlock4.cube']:
    v = view_orbital(orbName, mol, iso=0.02, alpha=0.9, html=True)

### Extract two-electron integrals within the frontier subspace

In [143]:
from pyscf import ao2mo
eri4mo = ao2mo.kernel(mol, mo_front_block)
eri4mo = ao2mo.restore(1, eri4mo, 4)

True

## Import vibronic coupling

In [ ]:
# --- Read electron-phonon matrix (convert to invcm) ---
#h5file = dir + '/dft-lvc-out.h5'
h5file = './dft-lvc-out.h5'
with h5py.File(h5file, 'r') as f:
    ephmat  = np.array(list(f['ephmat']))  * 219474.63
    omega   = np.array(list(f['omega']))   * 219474.63
    modevec = np.array(list(f['modevec']))

In [ ]:
# --- Convert ephmat to vibronic coupling in the localised subspace ---
g = np.einsum('pi, kij, jq -> kpq',
              mo_front_block.T,
              ephmat, 
              mo_front_block, 
              optimize=True)


# --- Convert modevec to Cartesian coordinates ---
modevec_cart = modevec.reshape(mol.natm, 3, modevec.shape[-1])
mass = mol.atom_mass_list()
for i in range(len(mass)):
    modevec_cart[i] = modevec_cart[i] / np.sqrt(mass[i])


# --- Vibronic coupling in the multiconfigurational basis LE+, LE-, CT+, CT-, TT ---
# index ordering hA  lA  hB  lB  (homo/lumo, left/right)
# index ordering SE+ SE- CT+ CT- TT
nvib = omega.shape[0]
W = np.zeros((nvib,5,5))
W[:,0,1] = 0.5 * ( g[:,2,2] - g[:,0,0] - g[:,3,3] + g[:,1,1] )
W[:,2,3] = 0.5 * ( g[:,2,2] - g[:,0,0] + g[:,3,3] - g[:,1,1] )
W[:,0,2] = g[:,1,3] - g[:,0,3]
W[:,1,3] = g[:,1,3] + g[:,0,3]
W[:,4,2] = np.sqrt(3) / 2 * ( g[:,1,2] + g[:,0,3] )
W[:,4,3] = np.sqrt(3) / 2 * ( g[:,1,2] - g[:,0,3] )
W = W + W.transpose(0,2,1)



In [114]:
#modevec_mw = np.einsum(' -> ', np.sqrt(mol.atom_mass_list()), modevec)

In [ ]:
a, k = 0.5, 0

mol_eq = lib.chkfile.load_mol('calc.chk')
xyz_eq = xyz_string(mol_eq.elements, mol_eq.atom_coords() * 0.529177)
xyz_pv = xyz_string(mol_eq.elements, 
                    (mol_eq.atom_coords() + a * modevec_cart[:,:,k]) * 0.529177)
xyz_nv = xyz_string(mol_eq.elements, 
                    (mol_eq.atom_coords() - a * modevec_cart[:,:,k]) * 0.529177)

view = py3Dmol.view(width=600, height=300)
view.clear()
view.addModel(xyz_eq, 'xyz')
view.addModel(xyz_pv, 'xyz')
view.addModel(xyz_nv, 'xyz')

#view.setStyle({'model':0}, {'stick':{'colorscheme':'grayCarbon', 'radius':0.05}})
view.setStyle({'model':1}, 
              {'stick':{'color':'red', 'radius':0.02},
               'sphere':{'color':'red', 'radius':0.1}})
view.setStyle({'model':2}, 
              {'stick':{'color':'blue', 'radius':0.02},
               'sphere':{'color':'blue', 'radius':0.1}})

view.setBackgroundColor('white')
view.zoomTo()
view.zoom(4)
view.show()


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
modevec[:,0].reshape(5,3)


array([[-0.17088917, -0.06495026, -0.31879499],
       [ 0.22580789, -0.19587707,  0.06432465],
       [ 0.08667406,  0.29427174, -0.24703139],
       [-0.1669471 ,  0.486007  ,  0.24237199],
       [-0.53105547, -0.03679971,  0.10063898]])

In [58]:
mol_eq.atom_coords()

array([[ 1.01011237e+00,  3.07518505e-01,  1.78764697e+00],
       [-3.54949429e-05, -1.44083742e-05, -4.80333262e-05],
       [ 3.91837382e-01,  1.57402052e+00, -1.29607767e+00],
       [ 6.38910390e-01, -1.77908112e+00, -8.58831615e-01],
       [-2.04101488e+00, -1.02537984e-01,  3.67024769e-01]])

AttributeError: 'Mole' object has no attribute 'size'

(666, 666)

## Idea:
### We already have the vibronic coupling in the full MO basis - let's use it

Instead of doing SVD to identify the vibronic coupling patterns in a small pre-selected electroinc state manifold, we could use the information from more than those four frontier molecular orbitals?

This is the same as identifying a preferred vibronic basis?
Identify hieararchies with SVD.